Current 4.0.1 numerical outputs and current-baseline Figure 4, Figure 5a and supplementary plotting cells. The reader does not replay 2.6.0 or generate old coefficient masks; figures use saved selected fits, aligned out-of-fold results and existing permutations without refitting.


In [ ]:
# Current4.0.1 numerical reader: select one explicit current candidate.
from pathlib import Path
import json
import numpy as np
import pandas as pd
from cca401_pipeline import load_final
from cca401.preferred import CURRENT
model = 'nc+fc'
volume_policy = "both"
master_seed = 1
outer_folds, inner_folds = 10, 5
branch = "B"
selection_path = CURRENT / "selection" / volume_policy / model.replace("+", "_") / f"seed_{master_seed}_{inner_folds}fold" / "selection.json"
selection = json.loads(selection_path.read_text())
choice = next(c for c in selection["choices"] if c["branch"] == branch)
fit_directory = CURRENT / choice["fit_directory"]
manifest, final_model = load_final(fit_directory)
print("Current full-data-CV-selected parameters:", choice["tau"])


In [ ]:
# Final full-sample fitted coefficients, with no coefficient filtering.
brain_loadings = final_model["brain"]
behavior_loadings = final_model["behavior"]
maps116 = pd.read_csv(fit_directory / "maps116.csv")
symptom_loadings = pd.read_csv(fit_directory / "symptom_loadings.csv", index_col=0)
display(symptom_loadings)
display(maps116)


In [ ]:
# OOF evaluation remains separate from the full-sample refit.
oof_directory = CURRENT / "oof" / volume_policy / model.replace("+", "_") / f"{outer_folds}x{inner_folds}" / f"seed_{master_seed}"
oof_summary = pd.read_csv(oof_directory / "summary.csv")
display(oof_summary[oof_summary.branch == branch])


### Figure 5a: cross-model consistency

Run the current baseline reader above before these cells. The comparison and drawing cells restore the original Figure 5a notebook source, using the saved current fits and aligned SDC input. The author-requested 0.10/ns centering is retained in the drawing cell. This does not refit any model.


In [ ]:
# Figure 5a inputs: saved full-sample fits and the agreed SDC CD2 display orientation.
from cca401 import preferred
from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests
import os
import matplotlib.pyplot as plt
import seaborn as sns
from plots import PAIR_COLOR, PANEL_TITLE_STYLE

with np.load(CURRENT / 'figure_inputs' / 'sdc_display_aligned.npz') as aligned:
    s_brain_ref = aligned['brain_ref'].copy()
    s_behav_ref = aligned['behavior_ref'].copy()
fdc_fit, _ = preferred.selected_fit('fc')
with np.load(fdc_fit / 'model.npz') as fdc:
    f_brain_ref = fdc['brain'].copy()
    f_behav_ref = fdc['behavior'].copy()
W_ref_x = final_model['brain']
U_behav_ref_fix = final_model['behavior']
figure5_output_dir = Path('figs/fig5')



In [ ]:
# 2. Define the current formal combined reference.
# ---------------------------------------------------------
comb_behav_ref = U_behav_ref_fix
comb_brain_ref = W_ref_x  # No PCA in the primary protocol.

n_brain = comb_brain_ref.shape[0]
if n_brain % 2 != 0:
    raise ValueError(f'Combined brain feature dimension should be even, got {n_brain}')

half = n_brain // 2
comb_brain_s_part = comb_brain_ref[:half, :]
comb_brain_f_part = comb_brain_ref[half:, :]

# ==========================================
# 3. 批量计算并存储结果
# ==========================================
n_dims_to_check = 2
results_list = []

# 定义对比的配对逻辑，方便循环
comparisons = [
    ('SDC vs FDC',             s_behav_ref,       f_behav_ref,       s_brain_ref,       f_brain_ref),
    ('Combined vs SDC',         comb_behav_ref,    s_behav_ref,       comb_brain_s_part, s_brain_ref),
    ('Combined vs FDC',         comb_behav_ref,    f_behav_ref,       comb_brain_f_part, f_brain_ref)
]

for dim in range(n_dims_to_check):
    for label, d1_behav, d2_behav, d1_brain, d2_brain in comparisons:
        
        # --- Behavior ---
        rb, pb = pearsonr(d1_behav[:, dim], d2_behav[:, dim])
        results_list.append({
            'Dim': dim + 1,
            'Pair': label,
            'Type': 'Behavior',
            'r': rb,
            'p_raw': pb
        })
        
        # --- Brain ---
        # 注意：这里假设 brain weights 维度是对齐的
        rbr, pbr = pearsonr(d1_brain[:, dim], d2_brain[:, dim])
        results_list.append({
            'Dim': dim + 1,
            'Pair': label,
            'Type': 'Brain',
            'r': rbr,
            'p_raw': pbr
        })

# ==========================================
# 4. 执行 FDR 校正
# ==========================================
# 提取所有原始 P 值
raw_p_values = [res['p_raw'] for res in results_list]

# 使用 Benjamini-Hochberg (fdr_bh) 方法进行校正
# alpha=0.05 是控制错误发现率的阈值
reject, p_adjusted, _, _ = multipletests(raw_p_values, alpha=0.05, method='fdr_bh')

# 将校正后的 P 值放回结果列表
for i, res in enumerate(results_list):
    res['p_adj'] = p_adjusted[i]
    res['significant'] = reject[i]

# ==========================================
# 5. 打印最终报表
# ==========================================
print(f"{'Dim':<4} | {'Comparison Pair':<20} | {'Type':<10} | {'Pearson r':<10} | {'P-raw':<10} | {'P-FDR':<10}")
print("=" * 85)

for res in results_list:
    # 格式化输出
    r_str = f"{res['r']:+.4f}"
    p_raw_str = f"{res['p_raw']:.2e}"
    p_adj_str = f"{res['p_adj']:.2e}"
    
    # 如果显著（FDR < 0.05），加个星号标记在最后
    sig_mark = "*" if res['p_adj'] < 0.05 else ""
    
    print(f"{res['Dim']:<4} | {res['Pair']:<20} | {res['Type']:<10} | {r_str:<10} | {p_raw_str:<10} | {p_adj_str:<10} {sig_mark}")

print("-" * 85)
print("P-FDR: Benjamini-Hochberg corrected p-values.")

In [ ]:
from matplotlib.lines import Line2D

# ==========================================
# 1. 自动转换数据 (List -> DataFrame)
# ==========================================
# 假设你上一步的分析结果存储在 'results_list' 变量中
# 我们需要把它变成 '宽格式' (Wide Format) 以便绘图

plot_data = []

# 获取所有的维度和对比对 (去重)
unique_dims = sorted(list(set(r['Dim'] for r in results_list)))
unique_pairs = sorted(list(set(r['Pair'] for r in results_list)), key=lambda x: ('SDC' in x, 'Combined' in x)) 
# 上面这个排序是为了保持: SDC vs FDC 在第一行 (根据具体字符串可能需要微调，或者直接硬编码顺序)

# 强制定义我们想要的顺序 (跟之前的图保持一致)
ordered_pairs = [
    'SDC vs FDC', 
    'Combined vs SDC', 
    'Combined vs FDC'
]

for dim in unique_dims:
    for pair in ordered_pairs:
        # 在 results_list 中查找对应的 Behavior 和 Brain 数据
        # 使用列表推导式找到匹配的字典
        behav_res = next((item for item in results_list if item['Dim'] == dim and item['Pair'] == pair and item['Type'] == 'Behavior'), None)
        brain_res = next((item for item in results_list if item['Dim'] == dim and item['Pair'] == pair and item['Type'] == 'Brain'), None)
        
        if behav_res and brain_res:
            plot_data.append({
                'Dimension': f'CD{dim}',
                'Comparison': pair,
                'Behavior_r': behav_res['r'],
                'Behavior_p': behav_res['p_adj'],  # 使用 FDR 校正后的 P 值
                'Brain_r':    brain_res['r'],
                'Brain_p':    brain_res['p_adj']   # 使用 FDR 校正后的 P 值
            })

df = pd.DataFrame(plot_data)

# 打印一下看看，确保转换正确
print("自动化生成的 DataFrame 前几行:")
print(df.head())

# ==========================================
# 2. 定义绘图辅助函数
# ==========================================
def get_stars(p):
    if p < 0.001: return '***'
    elif p < 0.01:  return '**'
    elif p < 0.05:  return '*'
    else:           return 'ns'  # 5.04e-02 会变成 ns

# ==========================================
# 3. 开始绘图 (包含所有视觉微调)
# ==========================================

save_path = str(figure5_output_dir / 'consistency_panel')

with plt.rc_context({'axes.linewidth': 1.2}):
    with sns.plotting_context("talk", font_scale=1.0), sns.axes_style("ticks"):
        color_behav = PAIR_COLOR[1][0]
        color_brain = PAIR_COLOR[1][1]

        fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.9), sharey=False)
        text_offset = 0.09 

        for i, dim in enumerate(['CD1', 'CD2']):
            ax = axes[i]
            # 筛选当前维度的数据
            subset = df[df['Dimension'] == dim].reset_index(drop=True)

            # 背景线
            ax.hlines(y=subset.index, xmin=0, xmax=1.0, color='gray', alpha=0.1, linewidth=1)

            # 连接线
            ax.hlines(y=subset.index, 
                      xmin=subset['Brain_r'], 
                      xmax=subset['Behavior_r'], 
                      color='gray', alpha=0.4, linewidth=2.5, zorder=1)

            # 画点
            ax.scatter(subset['Brain_r'], subset.index, color=color_brain, s=160, zorder=3, edgecolors='white', linewidth=0.5)
            ax.scatter(subset['Behavior_r'], subset.index, color=color_behav, s=160, zorder=3, edgecolors='white', linewidth=0.5)

            # --- 自动化标注文字 ---
            for idx, row in subset.iterrows():

                # 1. 准备文本 (自动读取 DataFrame 中的数值)
                brain_txt = f"{row['Brain_r']:.3f}\n{get_stars(row['Brain_p'])}"
                behav_txt = f"{row['Behavior_r']:.3f}\n{get_stars(row['Behavior_p'])}"

                # 2. 左右对齐策略
                ha_brain = 'right' if row['Brain_r'] > 0.9 else 'center'
                ha_behav = 'right' if row['Behavior_r'] > 0.9 else ('left' if row['Behavior_r'] < 0.1 else 'center')

                # 3. 上下位置策略 (Behavior在上, Brain在下)
                va_brain = 'top'
                y_brain = idx - text_offset

                va_behav = 'bottom'
                y_behav = idx + text_offset

                # 【特判】：最底下一行 (idx==0, SDC vs FDC)
                if idx == 0:
                    va_brain = 'bottom'
                    y_brain = idx + text_offset 

                # 绘制
                ax.text(row['Brain_r'], y_brain, brain_txt, 
                        ha=ha_brain, va=va_brain, color=color_brain, fontsize=11, fontweight='bold')

                ax.text(row['Behavior_r'], y_behav, behav_txt, 
                        ha=ha_behav, va=va_behav, color=color_behav, fontsize=11, fontweight='bold')

            # --- 坐标轴调整 ---
            ax.set_yticks(subset.index)
            if i == 0:
                ax.set_yticklabels(subset['Comparison'], fontsize=13, fontweight='bold')
            else:
                ax.set_yticklabels([])
                ax.tick_params(axis='y', length=0)

            ax.set_title(f'{dim} Consistency', fontsize=16, pad=15, fontweight='bold')

            ax.set_xlim(-0.05, 1.05) 
            ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
            ax.set_xlabel(r"Pearson correlation ($r$)", fontsize=15)
            ax.tick_params(axis='x', labelsize=12)

            sns.despine(ax=ax, left=True if i==1 else False, trim=True)
            ax.grid(axis='x', linestyle='--', alpha=0.3)

        # 图例
        legend_elements = [
            Line2D([0], [0], marker='o', color='w', markerfacecolor=color_brain, markersize=12, label='Brain loadings'),
            Line2D([0], [0], marker='o', color='w', markerfacecolor=color_behav, markersize=12, label='Behavior loadings'),
        ]
        fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 0.88), 
                   ncol=2, frameon=False, fontsize=15)

        fig.suptitle('Cross-model loading consistency', x=0.5, y=0.98,
                     ha='center', **{**PANEL_TITLE_STYLE, 'fontsize': 23.28})
        fig.subplots_adjust(left=0.20, right=0.98, top=0.70, bottom=0.16, wspace=0.18)
        if save_path:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            plt.savefig(f'{save_path}.pdf', bbox_inches='tight')
        plt.show()


### Figure 4: current baseline subpanels

Run the current numerical reader above first. These original plotting cells use saved current results; no model is refit.


In [ ]:
# Current-baseline data for the original Figure 4 plotting cells.
from cca401 import preferred
from cca401.nested_utils import SCALES_23
from plots import *
from pathlib import Path
from utils import get_mmpi_labels
from IPython.display import display

ROOT = Path.cwd()
figure_output_dir = ROOT / 'figs' / 'fig4'
scales = SCALES_23
sig_dims = [0, 1]
sig_idx_plot = np.array(sig_dims)
with np.load(preferred.oof_directory(model) / 'B_aligned.npz') as observed:
    U_fc_cv_fix = observed['brain'].copy()
    U_behav_cv_fix = observed['behavior'].copy()
    avg_fc_cv = observed['x_score'].copy()
    avg_behav_cv = observed['y_score'].copy()
U_fc_ref_fix = final_model['brain'].copy()
U_behav_ref_fix = final_model['behavior'].copy()
if model == 'nc':
    with np.load(CURRENT / 'figure_inputs' / 'sdc_display_aligned.npz') as aligned:
        U_fc_cv_fix = aligned['brain_cv'].copy()
        U_behav_cv_fix = aligned['behavior_cv'].copy()
        avg_fc_cv = aligned['x_score'].copy()
        avg_behav_cv = aligned['y_score'].copy()
        U_fc_ref_fix = aligned['brain_ref'].copy()
        U_behav_ref_fix = aligned['behavior_ref'].copy()
permutation_dir = ROOT / 'results' / 'baseline_followups_20261031' / 'permutations'
nulls = pd.read_csv(permutation_dir / 'permutation_nulls.csv')
model_nulls = nulls.loc[nulls['model'] == model].sort_values('permutation_index')
r_perm = model_nulls[['r_1', 'r_2']].to_numpy(float)
inference = pd.read_csv(permutation_dir / 'inference.csv')
r_cv = inference.loc[inference['model'] == model].sort_values('dimension')['observed_r'].to_numpy(float)

def resolve_save_path(key, stem):
    figure_output_dir.mkdir(parents=True, exist_ok=True)
    return str(figure_output_dir / stem)

save_path_brain_topk = resolve_save_path('brain_topk_loadings', 'brain_topk_loadings_combined')



In [ ]:
# ===== Brain Structure/Function Top Loadings via plot_loadings (no significance test) =====
from pathlib import Path


def load_atlas_labels(atlas_label_txt: str) -> list[str]:
    """Parse atlas label file where odd lines are labels and even lines are RGBA numbers."""
    p = Path(atlas_label_txt)
    if not p.exists():
        raise FileNotFoundError(f'Atlas label file not found: {p}')

    lines = [ln.strip() for ln in p.read_text().splitlines() if ln.strip()]
    labels = lines[::2]  # odd rows in 1-based indexing
    if len(labels) != 116:
        raise ValueError(f'Expected 116 atlas labels, got {len(labels)}')
    return labels


def split_brain_modalities(U_brain_ref: np.ndarray, U_brain_cv: np.ndarray) -> dict:
    """Split combined brain loadings (232=116 sc + 116 fc) into structure/function parts."""
    if U_brain_ref.ndim != 2:
        raise ValueError(f'U_brain_ref must be 2D, got shape={U_brain_ref.shape}')
    if U_brain_cv.ndim != 3:
        raise ValueError(f'U_brain_cv must be 3D, got shape={U_brain_cv.shape}')

    n_feat = U_brain_ref.shape[0]
    if n_feat != U_brain_cv.shape[1]:
        raise ValueError('Reference and CV brain feature counts do not match')
    if n_feat != 232:
        raise ValueError(f'Expected combined brain feature count=232, got {n_feat}')

    half = n_feat // 2
    return {
        'structure_ref': U_brain_ref[:half, :],
        'function_ref': U_brain_ref[half:, :],
        'structure_cv': U_brain_cv[:, :half, :],
        'function_cv': U_brain_cv[:, half:, :],
    }


def compute_dim_cv_summary(
    U_brain_cv: np.ndarray,
    dims: list[int],
) -> dict:
    """Per-dim CV mean/SD summary after the same normalization used in plot_loadings(cv=True)."""
    n_fold, n_feat, n_dims_all = U_brain_cv.shape
    dims = [int(d) for d in dims]

    mean_mat = np.zeros((n_feat, len(dims)), dtype=float)
    std_mat = np.zeros((n_feat, len(dims)), dtype=float)

    for j, d in enumerate(dims):
        if d < 0 or d >= n_dims_all:
            raise IndexError(f'dim index out of range: {d} (n_dims={n_dims_all})')

        data = U_brain_cv[:, :, d].copy()  # (fold, feat)

        # Match plot_loadings(cv=True): normalize by max absolute CV-mean in this dim.
        denom = np.abs(data.mean(axis=0)).max()
        if denom == 0:
            denom = 1.0
        data = data / denom

        mean_mat[:, j] = data.mean(axis=0)
        std_mat[:, j] = data.std(axis=0)

    return {
        'cv_mean_mat': mean_mat,
        'cv_std_mat': std_mat,
    }


def build_per_dim_topk_for_plot(
    U_brain_ref: np.ndarray,
    U_brain_cv: np.ndarray,
    atlas_labels: list[str],
    dims: list[int],
    summary_dict: dict,
    topk: int = 20,
):
    """Build one plotting tensor for plot_loadings with per-dim top-k by |CV-mean|."""
    n_atlas = len(atlas_labels)
    if U_brain_ref.shape[0] != 2 * n_atlas:
        raise ValueError('Combined brain feature count must equal 2 * atlas labels length')

    dims = [int(d) for d in dims]
    cv_mean_mat = summary_dict['cv_mean_mat']
    cv_std_mat = summary_dict['cv_std_mat']

    # Normalized ref loadings for table checks.
    ref_norm_cols = []
    for d in dims:
        col = U_brain_ref[:, d].copy()
        den = np.max(np.abs(col))
        if den == 0:
            den = 1.0
        ref_norm_cols.append(col / den)
    ref_norm = np.column_stack(ref_norm_cols)  # (232, n_dims)

    n_feat = U_brain_ref.shape[0]
    all_idx = np.arange(n_feat, dtype=int)

    selected_by_dim = []
    for j in range(len(dims)):
        rank_score = np.abs(cv_mean_mat[:, j])
        k = min(int(topk), int(n_feat))
        order = np.argsort(rank_score)[::-1][:k]
        selected_by_dim.append(all_idx[order])

    # Union of per-dim selected features, keeping dim/rank order.
    union_idx = []
    seen = set()
    for arr in selected_by_dim:
        for gi in arr.tolist():
            if gi not in seen:
                seen.add(gi)
                union_idx.append(int(gi))

    if len(union_idx) == 0:
        raise ValueError('No features selected for plotting.')

    union_idx = np.array(union_idx, dtype=int)

    # Prepare plotting tensor and zero-out non-top features per dim.
    cv_plot = U_brain_cv[:, union_idx][:, :, dims].copy()  # (fold, nunion, n_dims)

    pos_of = {int(g): i for i, g in enumerate(union_idx.tolist())}
    for j, sel in enumerate(selected_by_dim):
        keep = np.zeros(len(union_idx), dtype=bool)
        for g in sel.tolist():
            keep[pos_of[int(g)]] = True
        cv_plot[:, ~keep, j] = 0.0

    # Variant labels/colors from union index.
    variants = []
    variant_categories = []
    for gi in union_idx.tolist():
        if gi < n_atlas:
            feat_name = atlas_labels[gi]
            variants.append(feat_name)
            variant_categories.append('Structure')
        else:
            atlas_idx = gi - n_atlas
            feat_name = atlas_labels[atlas_idx]
            variants.append(feat_name)
            variant_categories.append('Function')

    # Long table: per-dim selected top-k.
    rows = []
    for j, d in enumerate(dims):
        for rank, gi in enumerate(selected_by_dim[j].tolist(), start=1):
            if gi < n_atlas:
                modality = 'structure'
                atlas_idx = gi
            else:
                modality = 'function'
                atlas_idx = gi - n_atlas

            rows.append({
                'dim': int(d) + 1,
                'rank_abs': int(rank),
                'global_feat_idx': int(gi),
                'modality': modality,
                'atlas_feat_idx': int(atlas_idx),
                'feat_name': atlas_labels[int(atlas_idx)],
                'cv_mean': float(cv_mean_mat[int(gi), j]),
            })

    top_tbl = pd.DataFrame(rows)
    return cv_plot, variants, variant_categories, top_tbl, selected_by_dim




# ----- 1) Keep existing sig_dims as target dimensions -----
if len(sig_dims) == 0:
    raise ValueError('sig_dims is empty. Brain plotting requires at least one canonical dimension.')

# ----- 2) Atlas label parsing (friendly labels) -----
atlas_path = ROOT / 'atlas' / 'Schaefer2018_100Parcels_7Networks_order_Tian_Subcortex_S1_label_friendly.txt'
atlas_labels = load_atlas_labels(str(atlas_path))
print(f'Atlas labels loaded: {len(atlas_labels)}')
print('Label sanity check:', atlas_labels[0], atlas_labels[15], atlas_labels[16])

# ----- 3) Split structure/function (validation only) -----
brain_parts = split_brain_modalities(U_fc_ref_fix, U_fc_cv_fix)
s_ref, f_ref = brain_parts['structure_ref'], brain_parts['function_ref']

print('Shape check U_fc_ref_fix:', U_fc_ref_fix.shape)
print('Structure/Function split:', s_ref.shape, f_ref.shape)
assert U_fc_ref_fix.shape[0] == 232
assert s_ref.shape[0] == 116 and f_ref.shape[0] == 116

# ----- 4) Per-dim top20 and one-figure plot (no significance filter) -----
TOPK = 20
dims_plot = [int(d) for d in sig_dims]

summary_dict = compute_dim_cv_summary(
    U_fc_cv_fix,
    dims=dims_plot,
)

cv_plot, variants_plot, variant_categories, brain_top20_table, selected_by_dim = build_per_dim_topk_for_plot(
    U_brain_ref=U_fc_ref_fix,
    U_brain_cv=U_fc_cv_fix,
    atlas_labels=atlas_labels,
    dims=dims_plot,
    summary_dict=summary_dict,
    topk=TOPK,
)

for j, d in enumerate(dims_plot):
    print(f'CD{d+1}: selected {len(selected_by_dim[j])} top features (max {TOPK})')
print('Selection rule per dim: rank by |CV-mean| only (no significance test)')
print(f'Union features used for one-figure plotting: {cv_plot.shape[1]}')

local_dims = list(range(len(dims_plot)))

plot_brain_loadings(
    cv_plot,
    dims=local_dims,
    title='',
    variants=variants_plot,
    variant_categories=variant_categories,
    category_colors={'Structure': PAIR_COLOR[0][0], 'Function': PAIR_COLOR[0][1]},
    category_order=['Structure', 'Function'],
    x_label=f'Top{TOPK} Brain Loadings (a.u.)',
    save_path=resolve_save_path('brain_topk_loadings', 'brain_topk_loadings'),
    legend_loc='upper left',
    network_palette=PALLETE,
    show_legend=True,
    show_network_legend=True,
    network_legend_each_axis=True,
    network_legend_loc='upper left',
    network_legend_ncol=1,
    network_legend_fontsize=15,
    network_legend_title=None,
    style_bars_by_network_and_modality=True,
    hollow_categories=('Structure',),
    hollow_bar_hatch='//',
    modality_legend_loc='lower right',
    modality_legend_fontsize=13,
    modality_legend_hollow_linewidth=1.1,
    modality_legend_each_axis=True,
)

print('brain_top20_table shape:', brain_top20_table.shape)

for dim_val in sorted(brain_top20_table['dim'].unique()):
    print(f'\n--- Top 20 Brain Loadings for Dimension {dim_val} ---')
    dim_table = brain_top20_table[brain_top20_table['dim'] == dim_val]
    display(dim_table.sort_values('rank_abs').head(20))


In [ ]:
behav_variants = get_mmpi_labels(keys=scales)

plot_loadings(
    U_behav_cv_fix,
    variants=behav_variants,
    dims=sig_dims,
    title='',
    legend_fontsize=13,
    legend_loc='auto',
    save_path=resolve_save_path('behav_cv_loadings', 'behav_cv_loadings'),
)

rows = []
for d in sig_dims:
    d = int(d)
    data = U_behav_cv_fix[:, :, d].copy()
    raw_mean = data.mean(axis=0)
    denom = np.abs(raw_mean).max()
    if denom == 0: denom = 1.0
    cv_mean_norm = raw_mean / denom
    
    order = np.argsort(np.abs(cv_mean_norm))[::-1]
    
    for rank, idx in enumerate(order, start=1):
        rows.append({
            'dim': d + 1,
            'rank_abs': rank,
            'feat_name': behav_variants[idx],
            'cv_mean': cv_mean_norm[idx]
        })

behav_loading_table = pd.DataFrame(rows)

for dim_val in sorted(behav_loading_table['dim'].unique()):
    print(f'\n--- All Behavior Loadings for Dimension {dim_val} ---')
    dim_table = behav_loading_table[behav_loading_table['dim'] == dim_val]
    display(dim_table.sort_values('rank_abs'))


In [ ]:
plot_correlation(
    avg_fc_cv,
    avg_behav_cv,
    dims=sig_dims,
    dot_size=20,
    title=None,
    x_label="Brain",
    save_path=resolve_save_path('brain_behav_corr', 'brain_behav_corr'),
)


In [ ]:
for color_index, dimension in enumerate(sig_idx_plot):
    fig, ax = plot_permutation_hist(
        r_perm,
        r_cv,
        idx=int(dimension),
        color=MULTI_COLOR[color_index % len(MULTI_COLOR)],
        save_path=resolve_save_path(
            "perm_hist",
            f"perm_hist_cd{int(dimension) + 1}",
        ),
    )
    plt.show()


In [ ]:
# ===== Brain network ring plots (Structure vs Function; joint normalization) =====
import re
from collections import OrderedDict
from pathlib import Path

NETWORKS = ['SC', 'VN', 'SMN', 'DAN', 'VAN', 'LN', 'FPCN', 'DMN']
atlas_path = ROOT / 'atlas' / 'Schaefer2018_100Parcels_7Networks_order_Tian_Subcortex_S1_label_friendly.txt'

if len(sig_dims) == 0:
    raise ValueError('sig_dims is empty. No canonical dimensions available for ring plotting.')

if U_fc_ref_fix.ndim != 2:
    raise ValueError(f'U_fc_ref_fix must be 2D, got shape={U_fc_ref_fix.shape}')
if U_fc_ref_fix.shape[0] != 232:
    raise ValueError(f'Expected U_fc_ref_fix feature dim=232 (116 structure + 116 function), got {U_fc_ref_fix.shape[0]}')

def _load_atlas_labels(path: Path) -> list[str]:
    if not path.exists():
        raise FileNotFoundError(f'Atlas label file not found: {path}')
    lines = [ln.strip() for ln in path.read_text().splitlines() if ln.strip()]
    labels = lines[::2]
    if len(labels) != 116:
        raise ValueError(f'Expected 116 atlas labels, got {len(labels)}')
    return labels

def _build_net_to_idx(labels: list[str], networks: list[str]) -> dict[str, list[int]]:
    pattern = re.compile(r'\(([A-Za-z0-9]+)/(?:L|R)\)\s*$')
    net_to_idx = {net: [] for net in networks}
    for i, label in enumerate(labels):
        m = pattern.search(label)
        if not m:
            raise ValueError(f'Cannot parse network token from atlas label: {label}')
        net = m.group(1).upper()
        if net not in net_to_idx:
            raise ValueError(f'Unknown network code {net} in atlas label: {label}')
        net_to_idx[net].append(i)
    missing = [net for net, idxs in net_to_idx.items() if len(idxs) == 0]
    if missing:
        raise ValueError(f'Networks with no ROI assignments: {missing}')
    return net_to_idx

atlas_labels = _load_atlas_labels(atlas_path)
net_to_idx = _build_net_to_idx(atlas_labels, NETWORKS)

print('Atlas/network check:', len(atlas_labels), 'labels;', {k: len(v) for k, v in net_to_idx.items()})
dims_for_ring = [int(d) for d in sig_dims]
if len(dims_for_ring) < 2:
    raise ValueError(f'Need at least 2 significant dimensions to save 4 ring PDFs, got {len(dims_for_ring)}')
dims_for_ring = dims_for_ring[:2]


# Network rings are displayed inline unless the global master switch is enabled.

for d in dims_for_ring:
    if d < 0 or d >= U_fc_ref_fix.shape[1]:
        raise IndexError(f'Dimension index out of range: {d} (n_dims={U_fc_ref_fix.shape[1]})')

    w = U_fc_ref_fix[:, d]
    w_s = np.abs(w[:116])
    w_f = np.abs(w[116:])

    s_mean = OrderedDict((net, float(np.mean(w_s[net_to_idx[net]]))) for net in NETWORKS)
    f_mean = OrderedDict((net, float(np.mean(w_f[net_to_idx[net]]))) for net in NETWORKS)

    den = float(sum(s_mean.values()) + sum(f_mean.values()))
    if den == 0:
        raise ValueError(f'CD{d+1}: denominator is 0 after abs+network-mean aggregation.')

    s_norm = OrderedDict((net, val / den) for net, val in s_mean.items())
    f_norm = OrderedDict((net, val / den) for net, val in f_mean.items())

    sum_struct = float(sum(s_norm.values()))
    sum_func = float(sum(f_norm.values()))
    sum16 = sum_struct + sum_func
    if not np.isclose(sum16, 1.0, atol=1e-8):
        raise AssertionError(f'CD{d+1}: sum16={sum16:.12f} != 1.0')

    print(f'CD{d+1}: sum16={sum16:.12f}, sum_struct={sum_struct:.12f}, sum_func={sum_func:.12f}')

    fig_s, ax_s = plt.subplots(figsize=(6, 6))
    plot_net_ring(
        s_norm,
        full_scale=.17,
        ring_width=0.095,
        ax=ax_s,
        title=f'Structure',
    )
    save_path_s = resolve_save_path('network_ring', f'ring_cd{d+1}_structure')

    if save_path_s:

        fig_s.savefig(f'{save_path_s}.pdf', bbox_inches='tight')

        print(f'Saved: {save_path_s}.pdf')
    plt.show()

    fig_f, ax_f = plt.subplots(figsize=(6, 6))
    plot_net_ring(
        f_norm,
        full_scale=.17,
        ring_width=0.095,
        ax=ax_f,
        title=f'Function',
    )
    save_path_f = resolve_save_path('network_ring', f'ring_cd{d+1}_function')

    if save_path_f:

        fig_f.savefig(f'{save_path_f}.pdf', bbox_inches='tight')

        print(f'Saved: {save_path_f}.pdf')
    plt.show()


In [ ]:
# Adopted Top 20 brain maps and shared colorbars, from the main plotting module.
plot_brain_maps(model, figure_output_dir)


### Supplementary sex and cross-fold agreement panels

Run the current Figure 4 data cell above first. The original drawing cells use the saved baseline scores and fitted loadings.


In [ ]:
# Inputs for the original supplementary sex and cross-fold agreement plots.
with np.load(preferred.oof_directory('nc+fc') / 'B_aligned.npz') as observed:
    score_ids = observed['ids'].astype(int)
demographics = pd.read_csv(ROOT / 'mmpi' / 'Iowa_MMPI2RF_byNandi.csv')
sex_by_id = demographics.loc[demographics['KEEP'] == 1].set_index('ID')['Sex']
sex_cv = sex_by_id.loc[score_ids].str.strip().str.upper().map({'M': 0, 'F': 1}).to_numpy(int)
W_ref_x, W_ref_y = final_model['brain'], final_model['behavior']
W_cv_x, W_cv_y = U_fc_cv_fix, U_behav_cv_fix
supp_gender_output_dir = ROOT / 'figs' / 'supp_gender'
supp_stability_output_dir = ROOT / 'figs' / 'supp_stability'
supp_gender_output_dir.mkdir(parents=True, exist_ok=True)
supp_stability_output_dir.mkdir(parents=True, exist_ok=True)



In [ ]:
dims_plot = [int(d) for d in sig_dims]
X_sig = avg_fc_cv[:, dims_plot]
Y_sig = avg_behav_cv[:, dims_plot]
title_sig = [f'CD{d+1}' for d in dims_plot]

plot_corr_with_label(
    X_sig,
    Y_sig,
    sex_cv,
    n_dims=len(dims_plot),
    dot_size=20,
    alpha=0.5,
    num_scales=1,
    label_mapping={0: 'Male', 1: 'Female'},
    colors=SUBTYPE_COLOR[:2][::-1],
    save_path=str(supp_gender_output_dir / 'brain_behav_corr_by_sex'),
)


In [ ]:
def compute_corr_with_ref(U_ref, U_cv):
    """
    calculate correlation between reference loading and 10-fold loading.

    Args:
        U_ref: shape (n_features, n_dims)
        U_cv: shape (n_folds, n_features, n_dims)
    
    Returns:
        corr_matrix: shape (n_dims, n_folds)
    """
    n_folds, _, n_dims = U_cv.shape
    corr = np.zeros((n_dims, n_folds))
    
    for i in range(n_dims):
        ref_vec = U_ref[:, i]
        for f in range(n_folds):
            fold_vec = U_cv[f, :, i]
            corr[i, f] = np.corrcoef(ref_vec, fold_vec)[0, 1]
    
    return corr

def plot_correlation_distribution(corr_data, title, save_path=None):
    """
    绘制单个数据集的相关系数分布（自适应组件数和图宽）
    """
    n_components = corr_data.shape[0]
    component_indices = np.arange(1, n_components + 1)

    # ---- figsize 自适应：13 个 → 8；线性缩放 ----
    width = 8 * (n_components / 13)
    fig, ax = plt.subplots(1, 1, figsize=(width, 5))

    ax.boxplot([corr_data[i, :] for i in range(n_components)],
               positions=component_indices,
               widths=0.6,
               patch_artist=True,
               boxprops=dict(facecolor='lightgray', alpha=0.7),
               medianprops=dict(color='black', linewidth=1.5),
               whiskerprops=dict(color='gray'),
               capprops=dict(color='gray'))

    medians = np.median(corr_data, axis=1)
    ax.plot(component_indices, medians, 'o--', color='black',
            linewidth=2, markersize=4, alpha=0.8)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.set_xlabel('Canonical component', fontsize=15)
    ax.set_ylabel('Cross-fold correlation with reference', fontsize=15)
    ax.set_title(title, fontsize=16, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_xticks(component_indices)
    ax.set_xticklabels([str(i) for i in component_indices])
    ax.tick_params(axis='both', labelsize=12)

    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(f'{save_path}.pdf', bbox_inches='tight')
    
# -------- corr_with_ref（新增，但保持默认 r_cv 逻辑不动） --------
corr_fc = compute_corr_with_ref(W_ref_x, W_cv_x)     # (eval_dim, nfold)
corr_symp = compute_corr_with_ref(W_ref_y, W_cv_y)   # (eval_dim, nfold)

plot_correlation_distribution(
    corr_fc,
    'Cross-validation agreement of brain loadings',
    save_path=str(supp_stability_output_dir / 'corr_fc_loading'),
)
plot_correlation_distribution(
    corr_symp,
    'Cross-validation agreement of behavior loadings',
    save_path=str(supp_stability_output_dir / 'corr_behav_loading'),
)
